# ESSE 4360 — AI for Payloads · Notebook 3
## Mapping floods with radar: Sentinel-1 SAR (and Sentinel-2) from Sen1Floods11

**What you will do**
1. Download real flood scenes from **Sen1Floods11** (Bonafilia et al., 2020): 512 × 512 chips at 10 m from 11 flood events, with **Sentinel-1** radar (VV, VH in dB), **Sentinel-2** optical, and **hand-labelled water** masks.
2. Map water with a **threshold** — no training at all.
3. Train a **random forest** per pixel on radar only, optical only, and both (data fusion).
4. Train a small **U-Net** that labels every pixel.
5. Score with **IoU** (intersection over union), per flood event.

**Why radar?** Calm water reflects the radar pulse away from the satellite, so it looks dark — day or night, through cloud.

**Runs on:** Colab or a laptop. Each chip is about 5 MB; the defaults download about 100 chips (~0.5 GB). A GPU makes the U-Net much faster.

In [ ]:
!pip -q install rasterio
import os, csv, time, random, zipfile, urllib.request, subprocess
import numpy as np, matplotlib.pyplot as plt, rasterio
import torch, torch.nn as nn, torch.nn.functional as Fn

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('Running on:', device)
DATA, FIG = 'sen1floods11', 'figures'; os.makedirs(DATA, exist_ok=True); os.makedirs(FIG, exist_ok=True)

N_TRAIN, N_VAL, N_TEST = 60, 15, 30          # chips to download from each official split
UNET_EPOCHS = 8 if device.type == 'cpu' else 30
BASE = 'https://storage.googleapis.com/sen1floods11/v1.1/'

## 1. Download the official split lists and a subset of chips
Files come from the public `sen1floods11` Google Cloud Storage bucket. If plain HTTPS fails, the cell tries `gsutil` (available in Colab).

In [ ]:
def fetch(rel, dest):
    if os.path.exists(dest): return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(BASE + rel, dest)
    except Exception as e:
        print('HTTPS failed (', e, ') - trying gsutil')
        subprocess.run(['gsutil', '-q', 'cp', 'gs://sen1floods11/v1.1/' + rel, dest], check=True)
    return dest

def chip_list(split):
    f = fetch(f'splits/flood_handlabeled/flood_{split}_data.csv', f'{DATA}/flood_{split}_data.csv')
    return [row[0].replace('_S1Hand.tif', '') for row in csv.reader(open(f)) if row]

splits = {s: chip_list(s) for s in ['train', 'valid', 'test']}
print({s: len(v) for s, v in splits.items()}, 'chips in the official hand-labelled splits')

HL = 'data/flood_events/HandLabeled/'
def get_chip(cid):
    s1 = fetch(f'{HL}S1Hand/{cid}_S1Hand.tif', f'{DATA}/S1/{cid}_S1Hand.tif')
    s2 = fetch(f'{HL}S2Hand/{cid}_S2Hand.tif', f'{DATA}/S2/{cid}_S2Hand.tif')
    lb = fetch(f'{HL}LabelHand/{cid}_LabelHand.tif', f'{DATA}/Label/{cid}_LabelHand.tif')
    s1a = np.nan_to_num(rasterio.open(s1).read().astype(np.float32), nan=-50)    # VV, VH (dB)
    s2a = rasterio.open(s2).read().astype(np.float32) / 10000.0                   # 13 bands, reflectance
    lba = rasterio.open(lb).read(1).astype(np.int16)                               # -1 no data, 0 land, 1 water
    return np.clip(s1a, -50, 1), s2a, lba

chips = {}
t0 = time.time()
for split, n in [('train', N_TRAIN), ('valid', N_VAL), ('test', N_TEST)]:
    pick = random.sample(splits[split], min(n, len(splits[split])))
    chips[split] = [(cid, *get_chip(cid)) for cid in pick]
    print(f'{split}: {len(chips[split])} chips  ({time.time() - t0:.0f} s so far)')

## 2. Look at a chip

In [ ]:
def stretch(a, p=(2, 98)):
    lo, hi = np.percentile(a, p); return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)

def rgb(s2): return np.dstack([stretch(s2[3]), stretch(s2[2]), stretch(s2[1])])

def show_chip(c, extra=None, fname=None):
    cid, s1, s2, lb = c
    panels = [('Sentinel-2 RGB', rgb(s2), None), ('S1 VV (dB)', s1[0], 'gray'), ('S1 VH (dB)', s1[1], 'gray'), ('hand label (water)', np.where(lb < 0, np.nan, lb), 'Blues')]
    if extra: panels += extra
    fig, ax = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.8))
    for a, (t, im, cm) in zip(ax, panels):
        kw = dict(vmin=-25, vmax=0) if 'VV' in t else dict(vmin=-32, vmax=-8) if 'VH' in t else {}
        a.imshow(im, cmap=cm, **kw); a.set_title(t, fontsize=10); a.axis('off')
    plt.suptitle(cid); plt.tight_layout()
    if fname: plt.savefig(f'{FIG}/{fname}', dpi=150)
    plt.show()

# the training chip with the most water
best = max(chips['train'], key=lambda c: (c[3] == 1).mean())
show_chip(best, fname='01_example_chip.png')

## 3. Metric: intersection over union (IoU)
For the water class: IoU = (pixels that are water in both the map and the truth) / (pixels that are water in either). It ignores the easy "dry land" majority, unlike accuracy. Pixels labelled −1 (no data) are ignored.

In [ ]:
def iou_counts(pred, lb):
    m = lb >= 0; p, t = pred[m] == 1, lb[m] == 1
    return (p & t).sum(), (p | t).sum()

def score(predict_fn, split='test', by_event=False):
    """predict_fn(s1, s2) -> 0/1 map. Returns pooled IoU (and per-event IoU)."""
    inter = union = 0; ev = {}
    for cid, s1, s2, lb in chips[split]:
        i, u = iou_counts(predict_fn(s1, s2), lb); inter += i; union += u
        e = cid.split('_')[0]; ev.setdefault(e, [0, 0]); ev[e][0] += i; ev[e][1] += u
    if by_event: return inter / union, {e: a / max(b, 1) for e, (a, b) in ev.items()}
    return inter / union

## 4. Method 1 — a threshold on VV (no training)
Pick one threshold from the training chips with Otsu's method (it splits a two-peaked histogram), then apply it everywhere.

In [ ]:
from skimage.filters import threshold_otsu
vv_train = np.concatenate([c[1][0][c[3] >= 0].ravel() for c in chips['train']])
vv_train = vv_train[vv_train > -40]
T = threshold_otsu(vv_train)
lab_train = np.concatenate([c[3][c[3] >= 0].ravel() for c in chips['train']])

fig, ax = plt.subplots(figsize=(7, 3.5))
v_all = np.concatenate([c[1][0][c[3] >= 0].ravel() for c in chips['train']])
ax.hist(v_all[lab_train == 0], 150, range=(-35, 5), alpha=0.6, label='not water', color='C1')
ax.hist(v_all[lab_train == 1], 150, range=(-35, 5), alpha=0.6, label='water', color='C0')
ax.axvline(T, color='k', ls='--', label=f'Otsu threshold {T:.1f} dB'); ax.set_xlabel('VV (dB)'); ax.legend(); ax.set_yscale('log')
plt.tight_layout(); plt.savefig(f'{FIG}/02_vv_histogram.png', dpi=150); plt.show()

thresh = lambda s1, s2: (s1[0] < T).astype(int)
results = {'Threshold on VV': score(thresh)}
print(f"Threshold IoU on test chips: {results['Threshold on VV']:.3f}")

## 5. Method 2 — random forest per pixel: radar, optical, and both
Each pixel is a feature vector. Radar: VV, VH, VV − VH. Optical: blue, green, red, NIR, SWIR-1, SWIR-2 and NDWI.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def pix_features(s1, s2, kind):
    f1 = [s1[0], s1[1], s1[0] - s1[1]]
    g, n = s2[2], s2[7]
    f2 = [s2[1], s2[2], s2[3], s2[7], s2[11], s2[12], (g - n) / (g + n + 1e-6)]
    f = {'S1': f1, 'S2': f2, 'S1+S2': f1 + f2}[kind]
    return np.stack(f, -1).reshape(-1, len(f))

def sample_pixels(kind, per_chip=1500):
    Xs, ys = [], []
    for cid, s1, s2, lb in chips['train']:
        Fm = pix_features(s1, s2, kind); l = lb.ravel(); ok = np.flatnonzero(l >= 0)
        k = np.random.choice(ok, min(per_chip, ok.size), replace=False); Xs.append(Fm[k]); ys.append(l[k])
    return np.concatenate(Xs), np.concatenate(ys)

rf_models = {}
for kind in ['S1', 'S2', 'S1+S2']:
    Xp, yp = sample_pixels(kind)
    rf = RandomForestClassifier(200, min_samples_leaf=5, n_jobs=-1, random_state=SEED).fit(Xp, yp); rf_models[kind] = rf
    results[f'Random forest, {kind}'] = score(lambda s1, s2, rf=rf, kind=kind: rf.predict(pix_features(s1, s2, kind)).reshape(s1.shape[1:]))
    print(f"Random forest {kind:6s} IoU {results[f'Random forest, {kind}']:.3f}")

> **Think:** optical bands often win on a clear day. But many flood images are cloudy — look at the Sentinel-2 RGB of a few test chips. Which method would you trust under cloud?

## 6. Method 3 — a small U-Net on the radar channels
The U-Net goes down (encoder), then up (decoder); skip connections carry fine detail across, so it outputs a label for every pixel. We train on random 256 × 256 crops with flips, and ignore no-data pixels in the loss.

In [ ]:
class UNet(nn.Module):
    def __init__(self, c_in=2, base=16):
        super().__init__()
        def cb(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.e1, self.e2, self.e3, self.b = cb(c_in, base), cb(base, 2 * base), cb(2 * base, 4 * base), cb(4 * base, 8 * base)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * base, 4 * base, 2, 2), cb(8 * base, 4 * base)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * base, 2 * base, 2, 2), cb(4 * base, 2 * base)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * base, base, 2, 2), cb(2 * base, base)
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(Fn.max_pool2d(e1, 2)); e3 = self.e3(Fn.max_pool2d(e2, 2)); b = self.b(Fn.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1)); d2 = self.d2(torch.cat([self.u2(d3), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

MU, SD = np.array([-11.0, -18.0], np.float32)[:, None, None], np.array([5.0, 5.0], np.float32)[:, None, None]
def norm_s1(s1): return (s1 - MU) / SD

def batches(split, bs=8, crop=256):
    data = chips[split]; idx = np.random.permutation(len(data))
    for i in range(0, len(idx), bs):
        xs, ys = [], []
        for k in idx[i:i + bs]:
            _, s1, _, lb = data[k]; r, c = np.random.randint(0, 512 - crop + 1, 2)
            x = norm_s1(s1)[:, r:r + crop, c:c + crop]; t = lb[r:r + crop, c:c + crop]
            if np.random.rand() < 0.5: x, t = x[:, :, ::-1], t[:, ::-1]
            if np.random.rand() < 0.5: x, t = x[:, ::-1], t[::-1]
            xs.append(x.copy()); ys.append(t.copy())
        yield torch.as_tensor(np.stack(xs)).to(device), torch.as_tensor(np.stack(ys)).to(device)

def masked_bce(logits, t):
    m = (t >= 0).float()
    return (Fn.binary_cross_entropy_with_logits(logits[:, 0], t.clamp(min=0).float(), reduction='none') * m).sum() / m.sum().clamp(min=1)

@torch.no_grad()
def unet_predict(model, s1):
    model.eval(); x = torch.as_tensor(norm_s1(s1)[None]).to(device)
    return (torch.sigmoid(model(x))[0, 0].cpu().numpy() > 0.5).astype(int)

unet = UNet().to(device); opt = torch.optim.Adam(unet.parameters(), lr=1e-3)
print('U-Net parameters:', sum(p.numel() for p in unet.parameters()))
hist = {'train': [], 'val_iou': []}
for ep in range(UNET_EPOCHS):
    unet.train(); t0 = time.time(); L = []
    for xb, tb in batches('train'):
        for _ in range(2):                       # two random crops per chip per epoch
            opt.zero_grad(); loss = masked_bce(unet(xb), tb); loss.backward(); opt.step(); L.append(loss.item())
    v = score(lambda s1, s2: unet_predict(unet, s1), split='valid')
    hist['train'].append(np.mean(L)); hist['val_iou'].append(v)
    print(f'epoch {ep + 1:2d}  loss {np.mean(L):.3f}  validation IoU {v:.3f}  ({time.time() - t0:.0f} s)')

results['U-Net, S1'] = score(lambda s1, s2: unet_predict(unet, s1))
print(f"U-Net test IoU: {results['U-Net, S1']:.3f}")

## 7. Compare — overall and per flood event

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(list(results), list(results.values()), color=['0.6', 'C0', 'C2', 'C4', 'C3'][:len(results)])
for i, v in enumerate(results.values()): ax.text(v + 0.01, i, f'{v:.2f}', va='center')
ax.set_xlim(0, 1); ax.set_xlabel('water IoU on test chips'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(f'{FIG}/03_method_comparison.png', dpi=150); plt.show()

_, ev_thr = score(thresh, by_event=True); _, ev_unet = score(lambda s1, s2: unet_predict(unet, s1), by_event=True)
_, ev_rf = score(lambda s1, s2: rf_models['S1+S2'].predict(pix_features(s1, s2, 'S1+S2')).reshape(s1.shape[1:]), by_event=True)
evs = sorted(ev_thr); xx = np.arange(len(evs))
fig, ax = plt.subplots(figsize=(10, 3.5))
for k, (d, n) in enumerate([(ev_thr, 'threshold'), (ev_rf, 'RF S1+S2'), (ev_unet, 'U-Net S1')]):
    ax.bar(xx + (k - 1) * 0.27, [d[e] for e in evs], 0.27, label=n)
ax.set_xticks(xx); ax.set_xticklabels(evs, rotation=45); ax.set_ylabel('IoU'); ax.legend(); ax.set_title('Per-event IoU (test chips)')
plt.tight_layout(); plt.savefig(f'{FIG}/04_per_event.png', dpi=150); plt.show()

In [ ]:
# Maps for the test chip with the most water
c = max(chips['test'], key=lambda c: (c[3] == 1).mean())
cid, s1, s2, lb = c
show_chip(c, extra=[('threshold', thresh(s1, s2), 'Blues'),
                    ('RF S1+S2', rf_models['S1+S2'].predict(pix_features(s1, s2, 'S1+S2')).reshape(512, 512), 'Blues'),
                    ('U-Net S1', unet_predict(unet, s1), 'Blues')], fname='05_test_maps.png')

## 8. Questions
1. Where does the threshold fail? (Look for radar shadow, smooth roads and tarmac, flooded vegetation, wind-roughened water.)
2. Which events score worst, and what do their images look like? Is the problem the data or the model?
3. Find a test chip where Sentinel-2 is cloudy. How do the optical and radar models compare there?
4. **Generalisation:** the paper behind this dataset kept one whole event (Bolivia) out of training to test generalisation. Remove all chips from one event from `chips['train']`, retrain, and score only that event. What happens?
5. **Payload question:** the U-Net has about 0.5 million parameters. Could it run on board to send down only a flood mask? Estimate the downlink saving for a 512 × 512 chip (2 bands × 32-bit floats versus a 1-bit mask).

In [ ]:
with zipfile.ZipFile('figures_notebook3.zip', 'w') as zz:
    for f in sorted(os.listdir(FIG)): zz.write(os.path.join(FIG, f))
try:
    from google.colab import files; files.download('figures_notebook3.zip')
except ImportError:
    print('Saved figures_notebook3.zip')